In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Input, GRU, Dense, Embedding, Attention
from tensorflow.keras.models import Model
import numpy as np
import json

In [ ]:
# 读取训练数据
data = []
with open('translation2019zh_train.json', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 10000: break  # 只读取前10000条数据
        data.append(json.loads(line))

print(data[0])
print(data[1])


In [ ]:
# 数据预处理
# 处理英文数据，将英文转小写， 并分词
en_texts = [d["english"].lower().split() for d in data]
# 将中文转成一个个字符
zh_texts = [list(d["chinese"]) for d in data]
print(en_texts)
print(zh_texts)

# 创建两个字典 作为英文和中文的词汇表
# 初始化4个特殊token
# <pad> 索引0， 用于填充序列，所有序列长度一致
# <sos> 索引1， 用于表示序列的开始
# <eos>: 索引2， 用于表示序列的结束
# <unk>: 索引3， 用于表示未知的token
en_vocab = {"<pad>":0, "<sos>":1, "<eos>":2, "<unk>":3}
zh_vocab = {"<pad>":0, "<sos>":1, "<eos>":2, "<unk>":3}

# 遍历英文和中文的文本， 构建词汇表
for en_text in en_texts:
    # 遍历英文句子中的每个单词
    for en_word in en_text:
        # 如果单词不再字典中， 则添加到字典中
        if en_word not in en_vocab:
            en_vocab[en_word] = len(en_vocab)

# 遍历中文的每句话
for zh_text in zh_texts:
    # 遍历中文句子中的每个单词
    for zh_word in zh_text:
        # 如果单词不再字典中， 则添加到字典中
        if zh_word not in zh_vocab:
            zh_vocab[zh_word] = len(zh_vocab)

print(en_vocab)
print(zh_vocab)

# 将英文与中文序列转为数字序列
# 将每句话转为索引序列， 前面加<sos> ，后面加<eos> , 查找单词的索引，如果不再词汇表中， 改词的token为<unk>
en_seqs = [[en_vocab["<sos>"]] + [en_vocab.get(t, en_vocab['<unk>']) for t in seq] + [en_vocab['<eos>']] for seq in en_texts]
zh_seqs = [[zh_vocab["<sos>"]] + [zh_vocab.get(t, en_vocab['<unk>']) for t in seq] + [zh_vocab['<eos>']] for seq in zh_texts]

print(en_seqs[0])
print(en_seqs[1])
print(zh_seqs)

# 将中文的数字序列处理为解码器的输入序列和目标序列
# 输入序列：去除结束符， 用于教师强制机制
decoder_input_seqs = [seq[:-1] for seq in zh_seqs]
# 目标序列：去掉开始符
decoder_target_seqs = [seq[1:] for seq in zh_seqs]

print(decoder_input_seqs[0])
print(decoder_target_seqs[0])

# 填充序列, 将所有序列的长度变为该类型的最大长度
# 计算最大长度
max_en_len = max([len(seq) for seq in en_seqs])
max_zh_input_len = max([len(seq) for seq in decoder_input_seqs])
max_zh_target_len = max([len(seq) for seq in decoder_target_seqs])
print(max_en_len)
print(max_zh_input_len)
print(max_zh_target_len)

# 填充序列
en_padded = tf.keras.preprocessing.sequence.pad_sequences(en_seqs, maxlen=max_en_len, padding='post')
decoder_input_seqs = tf.keras.preprocessing.sequence.pad_sequences(decoder_input_seqs, maxlen=max_zh_input_len, padding='post')
decoder_target_seqs = tf.keras.preprocessing.sequence.pad_sequences(decoder_target_seqs, maxlen=max_zh_target_len, padding='post')

print(en_padded[0])
print(decoder_input_seqs[0])
print(decoder_target_seqs[0])



In [ ]:
# 构建模型
embedding_dim = 128 # 词向量维度
units = 256 # 隐藏状态维度

# 构建编码器
# 输入层, None 表示任意长度， 即表示输入序列的长度是可变的
encoder_inputs = Input(shape=(None,))
# 词嵌入层
# Len(en_vocab) 表示词向量字典的大小
# embedding_dim 表示词向量的维度
# （encoder_input) 通过函数式调用将输入层链接到词嵌入层
enc_emb = Embedding(len(en_vocab), embedding_dim)(encoder_inputs)
# GRU层
# units 表示隐藏状态维度
# return_sequences = True 表示返回所有时间步的输出，用于注意力机制
# return_state =  True 表示返回最后一个时间步的隐藏状态，用于初始化解码器的隐藏状态
# (enc_emb)表示将嵌入层链接到GRU层
# 输出
# encoder_outputs：表示所有时间步的隐藏状态，用于注意力机制
# state_h： 表示最后一个时间步的隐藏状态， 用于初始化解码器的隐藏状态
encoder_outputs, state_h = GRU(units, return_state=True, return_sequences=True)(enc_emb)

# 解码器
decoder_inputs = Input(shape=(None,))
# 词嵌入层
dec_emb = Embedding(len(zh_vocab), embedding_dim)(decoder_inputs)
# GRU层
# initial_state=state_h 表示初始化解码器的隐藏状态为编码器的最后一个时间步骤的隐藏状态
# _:表示最后一个时间步的状态向量，_表示忽略，因为训练时不需要
decoder_outputs, _ = GRU(units, return_state=True, return_sequences=True)(dec_emb, initial_state=state_h)


#  注意力机制
# decoder_outputs: 解码器每个时间步的状态向量
# encoder_outputs: 编码器每个时间步的隐藏状态
context = Attention()([decoder_outputs, encoder_outputs])

# 将加权语义特征和状态向量拼接
# axis=-1，：制定拼接的轴， -1表示最后一个轴，即状态向量的维度
decoder_combined = tf.concat([context, decoder_outputs], axis=-1)

# 输出层
# 使用全连接层将拼接后的特征映射到中文词汇表大小的输出
# len(zh_vocab): 中文词汇的大小
# activation = 'softmax'：输出使用softmax函数，将输出转为概率分布
outputs = Dense(len(zh_vocab), activation='softmax')(decoder_combined)

# 定义完整的Seq2seq模型
# encoder_inputs: 编码器的输入层
# decoder_inputs: 解码器的输入层
# outputs： 中文输出序列
model = Model([encoder_inputs, decoder_inputs], outputs)

In [ ]:
# 模型训练
# 配置模型的训练参数
# optimizer='adam': 优化器选择为adam
# loss = 'sparse_categorical_crossentropy': 损失函数，适用于多分类任务
# metrics=['accuracy']: 评估指标，用于衡量训练过程中的准确率
model.compile(optimizer='adam', loss='spare_categorical_crossentropy', metrics=['accuracy'])

# 训练模型
# en_padded: 英文句子的填充后的序列
# decoder_input_seqs: 填充后的中文序列，去掉了最后一个字符，用于教师强制机制
# np.expand_dims(decoder_target_seqs, -1): 将目标序列的维度增加一维，以匹配模型的输出维度
# batch_size = 32: 批次大小，每次训练的样本数量
# epochs = 10: 训练轮数
# verbose=1: 显示训练进度
model.fit(
    [en_padded, decoder_input_seqs],
    np.expand_dims(decoder_target_seqs, -1),
    batch_size=32,
    epochs=10,
    verbose=1
)

In [2]:
#  测试模型
test_sentence = "I LOVE YOU"
# 1. 数据预处理
# 转小写，分词
test_sentence = test_sentence.lower().split()
# 将测试数据转为数字序列，符合模型输入格式
test_seq = [en_vocab['<sos>']] + [en_vocab.get(t, en_vocab['<unk>']) for t in test_sentence] + [en_vocab['<eos>']]
print(test_seq)
# 将序列填充到训练时的最大长度
test_padded = tf.keras.preprocessing.sequence.pad_sequences([test_seq], maxlen=max_en_len, padding='post')
print(test_padded)

# 2， 使用模型翻译
# 创建预测结果数组，初始化时一个全零的Numpy数组，，形状为(1, max_zh_target_len+1)
target_seq = np.zeros((1, max_zh_target_len + 1))
# 将第一个时间步设置为<sos>的索引，表示翻译开始
target_seq[0, 0] = zh_vocab['<sos>']

# 循环预测翻译
for t in range(max_zh_target_len):
    # 使用训练好的模型预测下一个字符
    # pred表示每个时间步的概率分布， verbose = 0，表示不输出预测过程
    pred = model.predict([test_padded, target_seq], verbose=0)
    # 取第0个样本（唯一样本）的第t个时间步的概率分布， 并取最大值对应的索引
    word_idx = np.argmax(pred[0, t])
    # 检查是否是<eos>, 如果是，则翻译结束
    if word_idx == zh_vocab['<sos>']:
        break
    # 将预测的索引添加到目标序列中
    target_seq[0, t+1] = word_idx

# 3. 将数字序列的翻译结果转为中文序列
# 创建一个反向词汇表， 将索引映射回字符
zh_inverse_vocab = {v: k for k, v in zh_vocab.items()}
# 将索引序列转为中文字符， 并过滤掉<pad> <sos>
translated = ''.join(zh_inverse_vocab.get(int(i), '') for i in target_seq[0][1: ] if i != 0)

# 输入
print(f"输入：{''.join(test_sentence)}")
print(f"翻译:{translated}")

NameError: name 'en_vocab' is not defined